# BraTS 2024 - Model Training

This notebook trains and evaluates two 2D segmentation models on preprocessed slice tensors (`E:bronikowski\data\processed_no_leaks`):
- **Part A:** U-Net (`best_unet.pt`)
- **Part B:** Attention U-Net (`best_attention_unet.pt`)

Inputs: 4 MRI channels (T1, T1CE, T2, FLAIR); outputs: 5 classes (0=background, 1=NCR, 2=ED, 3=ET, 4=RC). Loss: **DiceCELoss**, optimizer: **AdamW**. Both models use the same data loaders and hyperparameters, so results are directly comparable.

## 1. Environment Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Resolve repository root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

repo_src = PROJECT_ROOT / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch
import numpy as np
import matplotlib.pyplot as plt

from brain_mri.data.dataset import create_dataloader
from brain_mri.models.unet import UNet
from brain_mri.models.attention_unet import AttentionUNet
from brain_mri.models.losses import DiceCELoss
from brain_mri.training import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")
if device.type == "cuda":
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")

## 2. Initialize DataLoaders

> **Note for Windows users:** Recommended `num_workers` is 0 or 2. If you encounter any multiprocessing issues in the notebook, set `num_workers=0`.

In [ ]:
DATA_DIR = Path(r"E:\abronikowski\data")
train_dir = DATA_DIR / "processed_no_leaks" / "train"
val_dir = DATA_DIR / "processed_no_leaks" / "validate"

# Loader parameters
BATCH_SIZE = 32
NUM_WORKERS = 2  # Set to 0 if multiprocessing errors occur on Windows
PIN_MEMORY = torch.cuda.is_available()

train_loader = create_dataloader(
    split_dir=train_dir,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = create_dataloader(
    split_dir=val_dir,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"Train batches:      {len(train_loader)} (total samples: {len(train_loader.dataset)})")
print(f"Validation batches: {len(val_loader)} (total samples: {len(val_loader.dataset)})")

test_dir = DATA_DIR / "processed_no_leaks" / "test"
test_loader = create_dataloader(
    split_dir=test_dir,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
print(f"Test batches:       {len(test_loader)} (total samples: {len(test_loader.dataset)})")
checkpoint_dir = PROJECT_ROOT / "checkpoints"

EPOCHS = 30


# Part A: U-Net

## Model, Loss Function, and Trainer Initialization

In [ ]:
# U-Net: 4 input MRI modalities, 5 output classes (0: background, 1: NCR, 2: ED, 3: ET, 4: RC)
model = UNet(in_channels=4, num_classes=5, base_features=64)

criterion = DiceCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=checkpoint_dir,
    scheduler=scheduler,
    checkpoint_name="best_unet.pt"
)
print("Trainer initialized successfully.")

## Training Loop Execution

In [ ]:
history = trainer.fit(epochs=EPOCHS)

## Learning Curves

In [ ]:
plt.figure(figsize=(14, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss During Training")
plt.legend()
plt.grid(True)

# Validation Mean Dice curves
plt.subplot(1, 2, 2)
plt.plot(history["val_mean_dice"], label="Val Mean Dice", color="green", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Mean Dice Score")
plt.legend()
plt.grid(True)

plt.tight_layout()
curve_save_path = checkpoint_dir / "learning_curves.png"
plt.savefig(curve_save_path, dpi=150)
print(f"Saved curve plot to: {curve_save_path}")
plt.show()

## Test Set Evaluation

Loads the best checkpoint (selected on validation Dice) and evaluates it once on the held-out test split.

In [ ]:
# Restore best weights (the model in memory holds the last epoch, not the best one)
checkpoint = torch.load(checkpoint_dir / "best_unet.pt", map_location=device, weights_only=True)
# Rebuild the matching architecture so the cell works regardless of which part ran last
model = UNet(in_channels=4, num_classes=5, base_features=64).to(device, memory_format=torch.channels_last)
model.load_state_dict(checkpoint["model_state_dict"])
trainer.model = model
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val Dice: {checkpoint['best_dice']:.4f})")

test_loss, test_dice = trainer.validate_epoch(test_loader)

print(f"\nTest Loss:      {test_loss:.4f}")
print(f"Test Mean Dice: {test_dice['mean']:.4f}")
for cls_name, score in test_dice.items():
    if cls_name != "mean":
        print(f"  {cls_name}: {score:.4f}")

### Per-class Dice: Validation vs Test

In [ ]:
import numpy as np
from brain_mri.models.metrics import CLASS_NAMES

# Val Dice of the same (best) checkpoint, for comparison with test
_, val_dice = trainer.validate_epoch()

classes = list(CLASS_NAMES.values()) + ["mean"]
x = np.arange(len(classes))
width = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width / 2 - 0.01, [val_dice[c] for c in classes], width, label="Validation", color="#2a78d6")
test_bars = ax.bar(x + width / 2 + 0.01, [test_dice[c] for c in classes], width, label="Test", color="#eb6834")
ax.bar_label(test_bars, fmt="%.2f", padding=3, fontsize=9, color="#3d3d3a")

ax.set_xticks(x, classes)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Dice score")
ax.set_title("Dice per class: validation vs test (best checkpoint)")
ax.yaxis.grid(True, color="#e5e5e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.savefig(checkpoint_dir / "test_dice.png", dpi=150)
plt.show()

### Example Segmentations

One slice per random test patient (largest tumor area). Last column: prediction as fill, ground truth as contour.

In [ ]:
from collections import defaultdict
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

N_PATIENTS = 4
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # NCR, ED, ET, RC
mask_cmap = ListedColormap(CLASS_COLORS)

test_ds = test_loader.dataset
slices_by_patient = defaultdict(list)
for i, f in enumerate(test_ds.files):
    slices_by_patient[f.parent.name].append(i)

# Random patients; for each, the slice with the largest tumor area
rng = np.random.default_rng(42)
patients = rng.choice(sorted(slices_by_patient), size=N_PATIENTS, replace=False)
samples = []
for pid in patients:
    best_idx = max(slices_by_patient[pid], key=lambda i: (test_ds[i][1] > 0).sum().item())
    samples.append((pid, *test_ds[best_idx]))

model.eval()
with torch.no_grad():
    images = torch.stack([s[1] for s in samples]).to(device)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        preds = model(images).argmax(dim=1).cpu()

def show_mask(ax, mask):
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=mask_cmap, vmin=1, vmax=4, alpha=0.55, interpolation="nearest")

fig, axes = plt.subplots(N_PATIENTS, 4, figsize=(14, 3.6 * N_PATIENTS))
for row, ((pid, image, gt), pred) in enumerate(zip(samples, preds)):
    flair = image[3].numpy()  # channel order: T1, T1CE, T2, FLAIR
    gt, pred = gt.numpy(), pred.numpy()
    wt_dice = 2 * ((gt > 0) & (pred > 0)).sum() / max((gt > 0).sum() + (pred > 0).sum(), 1)

    for ax in axes[row]:
        ax.imshow(flair, cmap="gray")
        ax.axis("off")
    show_mask(axes[row, 1], gt)
    show_mask(axes[row, 2], pred)
    # Prediction as fill, ground truth as contours on the same image
    show_mask(axes[row, 3], pred)
    for cls, color in zip(CLASS_NAMES, CLASS_COLORS):
        if (gt == cls).any():
            axes[row, 3].contour(gt == cls, levels=[0.5], colors=[color], linewidths=1.2)

    axes[row, 0].set_title(f"{pid}\nFLAIR", fontsize=10)
    axes[row, 1].set_title("Ground truth", fontsize=10)
    axes[row, 2].set_title("Prediction", fontsize=10)
    axes[row, 3].set_title(f"Prediction (fill) + GT (contour)\nwhole-tumor Dice: {wt_dice:.2f}", fontsize=10)

handles = [Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES.values())]
handles.append(Line2D([], [], color="#3d3d3a", linewidth=1.2, label="GT contour"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=11)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig(checkpoint_dir / "test_examples.png", dpi=150)
plt.show()

# Part B: Attention U-Net

## Model, Loss Function, and Trainer Initialization

In [ ]:
# Attention U-Net: 4 input MRI modalities, 5 output classes (0: background, 1: NCR, 2: ED, 3: ET, 4: RC)
model = AttentionUNet(in_channels=4, num_classes=5, base_features=64)

criterion = DiceCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=checkpoint_dir,
    scheduler=scheduler,
    checkpoint_name="best_attention_unet.pt"
)
print("Trainer initialized successfully.")

## Training Loop Execution

In [ ]:
history = trainer.fit(epochs=EPOCHS)

## Learning Curves

In [ ]:
plt.figure(figsize=(14, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss During Training")
plt.legend()
plt.grid(True)

# Validation Mean Dice curves
plt.subplot(1, 2, 2)
plt.plot(history["val_mean_dice"], label="Val Mean Dice", color="green", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Mean Dice Score")
plt.legend()
plt.grid(True)

plt.tight_layout()
curve_save_path = checkpoint_dir / "learning_curves_attention.png"
plt.savefig(curve_save_path, dpi=150)
print(f"Saved curve plot to: {curve_save_path}")
plt.show()

## Test Set Evaluation

Loads the best checkpoint (selected on validation Dice) and evaluates it once on the held-out test split.

In [ ]:
# Restore best weights (the model in memory holds the last epoch, not the best one)
checkpoint = torch.load(checkpoint_dir / "best_attention_unet.pt", map_location=device, weights_only=True)
# Rebuild the matching architecture so the cell works regardless of which part ran last
model = AttentionUNet(in_channels=4, num_classes=5, base_features=64).to(device, memory_format=torch.channels_last)
model.load_state_dict(checkpoint["model_state_dict"])
trainer.model = model
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val Dice: {checkpoint['best_dice']:.4f})")

test_loss, test_dice = trainer.validate_epoch(test_loader)

print(f"\nTest Loss:      {test_loss:.4f}")
print(f"Test Mean Dice: {test_dice['mean']:.4f}")
for cls_name, score in test_dice.items():
    if cls_name != "mean":
        print(f"  {cls_name}: {score:.4f}")

### Per-class Dice: Validation vs Test

In [ ]:
import numpy as np
from brain_mri.models.metrics import CLASS_NAMES

# Val Dice of the same (best) checkpoint, for comparison with test
_, val_dice = trainer.validate_epoch()

classes = list(CLASS_NAMES.values()) + ["mean"]
x = np.arange(len(classes))
width = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width / 2 - 0.01, [val_dice[c] for c in classes], width, label="Validation", color="#2a78d6")
test_bars = ax.bar(x + width / 2 + 0.01, [test_dice[c] for c in classes], width, label="Test", color="#eb6834")
ax.bar_label(test_bars, fmt="%.2f", padding=3, fontsize=9, color="#3d3d3a")

ax.set_xticks(x, classes)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Dice score")
ax.set_title("Dice per class: validation vs test (best checkpoint)")
ax.yaxis.grid(True, color="#e5e5e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.savefig(checkpoint_dir / "test_dice_attention.png", dpi=150)
plt.show()

### Example Segmentations

One slice per random test patient (largest tumor area). Last column: prediction as fill, ground truth as contour.

In [ ]:
from collections import defaultdict
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

N_PATIENTS = 4
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # NCR, ED, ET, RC
mask_cmap = ListedColormap(CLASS_COLORS)

test_ds = test_loader.dataset
slices_by_patient = defaultdict(list)
for i, f in enumerate(test_ds.files):
    slices_by_patient[f.parent.name].append(i)

# Random patients; for each, the slice with the largest tumor area
rng = np.random.default_rng(42)
patients = rng.choice(sorted(slices_by_patient), size=N_PATIENTS, replace=False)
samples = []
for pid in patients:
    best_idx = max(slices_by_patient[pid], key=lambda i: (test_ds[i][1] > 0).sum().item())
    samples.append((pid, *test_ds[best_idx]))

model.eval()
with torch.no_grad():
    images = torch.stack([s[1] for s in samples]).to(device)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        preds = model(images).argmax(dim=1).cpu()

def show_mask(ax, mask):
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=mask_cmap, vmin=1, vmax=4, alpha=0.55, interpolation="nearest")

fig, axes = plt.subplots(N_PATIENTS, 4, figsize=(14, 3.6 * N_PATIENTS))
for row, ((pid, image, gt), pred) in enumerate(zip(samples, preds)):
    flair = image[3].numpy()  # channel order: T1, T1CE, T2, FLAIR
    gt, pred = gt.numpy(), pred.numpy()
    wt_dice = 2 * ((gt > 0) & (pred > 0)).sum() / max((gt > 0).sum() + (pred > 0).sum(), 1)

    for ax in axes[row]:
        ax.imshow(flair, cmap="gray")
        ax.axis("off")
    show_mask(axes[row, 1], gt)
    show_mask(axes[row, 2], pred)
    # Prediction as fill, ground truth as contours on the same image
    show_mask(axes[row, 3], pred)
    for cls, color in zip(CLASS_NAMES, CLASS_COLORS):
        if (gt == cls).any():
            axes[row, 3].contour(gt == cls, levels=[0.5], colors=[color], linewidths=1.2)

    axes[row, 0].set_title(f"{pid}\nFLAIR", fontsize=10)
    axes[row, 1].set_title("Ground truth", fontsize=10)
    axes[row, 2].set_title("Prediction", fontsize=10)
    axes[row, 3].set_title(f"Prediction (fill) + GT (contour)\nwhole-tumor Dice: {wt_dice:.2f}", fontsize=10)

handles = [Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES.values())]
handles.append(Line2D([], [], color="#3d3d3a", linewidth=1.2, label="GT contour"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=11)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig(checkpoint_dir / "test_examples_attention.png", dpi=150)
plt.show()